# DELTA LAKE

Local instance to manage MatrizActividades

In [1]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import re
import json
import pickle
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)


# Ubicación del directorio DELTA LAKE TABLE
table_path = "./test/deltalake_2025"

# New Delta Lake
table_v23 = "/home/vlad/deltav23"

Success!!!


In [2]:
# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


In [48]:
# DELTA LAKE Connection
# Verify the existence of the DELTA LAKE table
import pandas as pd
import numpy as np
from deltalake import DeltaTable
from datetime import datetime
from pprint import pprint

if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    dt = DeltaTable(table_path)
    df = dt.to_pandas()
    print(f"Conectado a la tabla Delta Lake en: {table_path}")



Conectado a la tabla Delta Lake en: ./test/deltalake_2025


### Conversion de Fechas en Pandas Dataframe

#### Funcion para eliminar del Text el componente de UTC TimeZone

In [43]:
def borra_time_zone( fecha ):
  """
  Esta función elimina el componenete de Time Zone y deja solamente la fecha y hora. 
  En caso de que no contenga este componente deja el String intacto. 
  """
  if not isinstance( fecha, str ):
    return pd.NaT
  
  if len(fecha) < 4:
    return pd.NaT

  fecha_inicio = fecha.replace('T', ' ').split()
  fecha_inicio = fecha_inicio[0]+' '+fecha_inicio[-1]
  return fecha_inicio

#### Funcion para determinar si el texto de Fecha es valido 

In [ ]:
def is_valid_utc_format(text_input):
    """
    Checks if a string can be converted to a timezone-aware datetime.

    The function tests against a specific ISO 8601 format that includes a
    UTC offset, like "2024-05-31T00:00:00-05:00".

    Args:
        text_input: The string or value to check.

    Returns:
        - True: if the input is a string and matches the format.
        - False: if the input is not a string or does not match the format.
        - pd.NaT: if the input is a null-like value (e.g., None, np.nan).
    """
    # 1. Handle null-like inputs first
    if pd.isna(text_input):
        return pd.NaT

    # 2. Ensure the input is a string before attempting to parse
    if not isinstance(text_input, str):
        return False

    # 3. Try to parse the string using the specific format
    try:
        # The format string matches the user's example.
        # %Y: 4-digit year
        # %m: 2-digit month
        # %d: 2-digit day
        # T: Literal 'T' separator
        # %H:%M:%S: Hour, minute, second
        # %z: UTC offset (e.g., -0500). Pandas extends this to handle
        #     the colon format (-05:00) as well.
        # errors='raise' ensures that any parsing failure raises an exception.
        pd.to_datetime(text_input, format="%Y-%m-%d %H:%M:%S", errors='raise')
        return True
    except ValueError:
        # This exception is raised if the string does not match the format.
        return False


#### Convertir el Texto de las columna 'Fecha', 'InicioEvento', 'FinEvento'

Convierte a un formato uniforme todas las Fechas

In [49]:
df['Fecha'] = df['Fecha'].apply( lambda x: borra_time_zone(x))
df['InicioEvento'] = df['InicioEvento'].apply( lambda x: borra_time_zone(x))
df['FinEvento'] = df['FinEvento'].apply( lambda x: borra_time_zone(x))

In [51]:
df["Fecha"] = pd.to_datetime(df["Fecha"])
df["InicioEvento"] = pd.to_datetime(df["InicioEvento"])  #, format='mixed'
df["FinEvento"] = pd.to_datetime(df["FinEvento"])

df.to_pickle("/home/vlad/Documents/mongodb_v23.pkl")

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234013 entries, 0 to 234012
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype                    
---  ------         --------------   -----                    
 0   Item           234013 non-null  int64                    
 1   Cuenta         234013 non-null  object                   
 2   Evento         234013 non-null  object                   
 3   Actividad      233705 non-null  object                   
 4   Alimentador    169973 non-null  object                   
 5   Primario       234013 non-null  object                   
 6   Desconexion    234013 non-null  object                   
 7   SIG            234013 non-null  object                   
 8   Tipo           220860 non-null  object                   
 9   Materiales     234013 non-null  object                   
 10  Cuadrilla      234013 non-null  object                   
 11  Dia            234013 non-null  object                   
 12  Fe

### Recargar Librerias Dinámicamente


In [4]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [5]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

## Verificacion de OT desde MongoDB hacia DeltaLake

### Cursor para obtener todos los "id_ot" desde MongoDB

In [30]:
""" 
   OBTENER TODOS LOS 'id_ot' desde MongoDB
"""

try:
    # 1. Use a projection to only retrieve the 'id_ot' field.
    #    - {'id_ot': 1} means "include this field".
    #    - {'_id': 0} means "exclude the default _id field".
    cursor = CurrentCollection.find({}, {'id_ot': 1, '_id': 0})

    # 2. Create a list from the cursor results using a list comprehension.
    #    This iterates through each document in the cursor and extracts 'id_ot'.
    id_ot_list = [doc['id_ot'] for doc in cursor]

    # 3. Now you have your list of all 'id_ot' values.
    print(f"Successfully retrieved {len(id_ot_list)} 'id_ot' values.")
    if id_ot_list:
        print("First 10 values:", id_ot_list[:10])

except Exception as e:
    print(f"An error occurred: {e}")


Successfully retrieved 21941 'id_ot' values.
First 10 values: [148859, 150217, 148664, 148843, 149271, 149150, 155468, 155428, 155505, 155585]


In [31]:
"""
   Obtener todos los 'id_ot' existentes en DeltaLake
"""

delta_ids = df["id_ot"].unique()
len(delta_ids)

21941

In [32]:
"""
   Difentecia de las ot que faltan en DeltaLake
"""
set_mongo = set(id_ot_list)
set_delta = set(delta_ids)

# Find which items in set_delta are not in set_mongo
new_ids_set = set_mongo.difference(set_delta)

# Convert the result back to a list
new_ids_to_process = list(new_ids_set)

print(f"Found {len(new_ids_to_process)} new IDs to be processed.")
# We sort the list here just for a predictable, clean output
print(f"New IDs: {sorted(new_ids_to_process)}")

Found 0 new IDs to be processed.
New IDs: []


In [ ]:
"""
   Descargar y procesar las OT faltantes y añadirlas al Delta Lake
"""
new_data_frames = []
for ot in new_ids_to_process:
  json_ot = CurrentCollection.find_one({"id_ot": ot})
  if not json_ot:
    print(f"No se pudo encontrar la OT con id_ot '{ot}' en MongoDB. Saltando.")
    continue
                
  obj_ot = OrdenTrabajo.GestionOt.from_dict(json_ot)
  new_data_frames.append(Actividades.ConvertirOT_a_ActividadesCSV(obj_ot))

In [97]:
try:
  new_df = pd.concat(new_data_frames, ignore_index=True)
  write_deltalake(table_path, new_df, mode='append')
  print(f" [ EXITO ] DELTA LAKE Se han añadido {len(new_df)} filas a la tabla Delta en '{table_path}'.")
except Exception as e:
  print(f"Fallo al escribir en la tabla Delta: {e}")

 [ EXITO ] DELTA LAKE Se han añadido 121556 filas a la tabla Delta en './test/deltalake_2025'.


## FULL MONGO DOWNLOAD

Generar un nuevo archivo Delta Lake para unificar versiones - Ejecutado JULIO 2025

### Descarga de OT's desde MongoDB hacia Pickle y Delta Lake 

In [43]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

# ... setup client, db, collection
cursor = CurrentCollection.find()
all_documents = cursor.to_list() 
# or simply: all_documents = list(cursor)
print(f"Loaded {len(all_documents)} documents into a list.")
client.close()



Loaded 21879 documents into a list.


In [ ]:
obj_list = []
for document in all_documents:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot) )
df = pd.concat(obj_list, ignore_index=True)


In [47]:
write_deltalake("/home/vlad/delta_v23", df)

In [4]:
dt = DeltaTable("/home/vlad/delta_v23")

## Descargar OT faltantes desde MongoDB hacia DeltaLake

### Cargar Pickle para analisis

In [ ]:
# Perform the merge operation
print("\n--- Merging changes back into Delta Table ---")

(
    dt.merge(
        source=df,
        predicate="target.id = source.id",
        source_alias="source",
        target_alias="target"
    )
    .when_matched_update_all()  # If id matches, update the row
    .when_not_matched_insert_all()  # If a new id is in the source, insert it
    .execute()
)

print("Merge complete.")

### ¿Son todas los items en 'Fecha' validos?

In [110]:
def is_valid_utc_format(text_input):
    """
    Checks if a string can be converted to a timezone-aware datetime.

    The function tests against a specific ISO 8601 format that includes a
    UTC offset, like "2024-05-31T00:00:00-05:00".

    Args:
        text_input: The string or value to check.

    Returns:
        - True: if the input is a string and matches the format.
        - False: if the input is not a string or does not match the format.
        - pd.NaT: if the input is a null-like value (e.g., None, np.nan).
    """
    # 1. Handle null-like inputs first
    if pd.isna(text_input):
        return pd.NaT

    # 2. Ensure the input is a string before attempting to parse
    if not isinstance(text_input, str):
        return False

    # 3. Try to parse the string using the specific format
    try:
        # The format string matches the user's example.
        # %Y: 4-digit year
        # %m: 2-digit month
        # %d: 2-digit day
        # T: Literal 'T' separator
        # %H:%M:%S: Hour, minute, second
        # %z: UTC offset (e.g., -0500). Pandas extends this to handle
        #     the colon format (-05:00) as well.
        # errors='raise' ensures that any parsing failure raises an exception.
        pd.to_datetime(text_input, format="%Y-%m-%d %H:%M:%S", errors='raise')
        return True
    except ValueError:
        # This exception is raised if the string does not match the format.
        return False



In [70]:
import datetime
# 2. Define a function to safely get the date
def safe_to_date(value):
    # Check if the value is a Timestamp or datetime object
    if isinstance(value, (pd.Timestamp, datetime.datetime)):
        return value.date()
    # If it's already a date object, just return it
    elif isinstance(value, datetime.date):
        return value
    # For any other type, return NaT (Not a Time)
    else:
        return pd.NaT

In [ ]:
df['dates_equal_Inicio'] = (df['Fecha'].dt.date == df['InicioEvento'].apply(safe_to_date))
df['dates_equal_Fin'] = (df['Fecha'].dt.date == df['FinEvento'].apply(safe_to_date))

In [58]:
df['dates_equal_Inicio'].unique()

array([ True, False])

In [65]:
falla_inicio = df.query("dates_equal_Inicio == False")
#falla_inicio[["Item","Responsable","id_ot","Fecha","InicioEvento","FinEvento"]]
falla_inicio["id_ot"].unique()

array([155505, 155762, 145572, 144647, 148359, 147423, 148549, 148277,
       148246, 146584, 147455, 147804, 148333, 149748, 149726, 149242,
       150570, 137497, 139625, 138768, 142840, 139823, 139690, 134776,
       137787, 128220, 139615, 144309, 141353, 144417, 134989, 143002,
       140128, 134479, 137253, 125688, 110252, 105181, 114346, 119028,
       116773, 120598, 114096, 102758, 106048, 115879, 113463, 100339,
       104339, 111938, 120285, 114645, 109933, 111049,  84875,  89820,
        85621,  96068,  80898,  83812,  82240,  92859,  81700,  86257,
        83222, 156391,  75212,  67725,  76277,  73194,  70460,  76774,
        67486,  60815,  64571,  54263,  50513])

### Buscamos docuementos faltantes en Mongo DB

In [19]:
# 2. Obtenemos los id_OT para compararlos con la base de datos en MongoDB

# Step 1: Collect all the IDs from your local list into a new list.
all_ids = [ot.id_ot for ot in obj_lists_dask]

# Step 2: Use the "$in" operator to find all documents in the database
# that match any ID in your list. This is ONE efficient query.
existing_docs_cursor = CurrentCollection.find(
    {"id_ot": {"$in": all_ids}},
    {"id_ot": 1}  # Projection: only return the _id and id_ot fields for efficiency
)

# Step 3: Create a set of the IDs that were actually found in the database.
# Sets provide very fast lookups.
ids_in_db = {doc['id_ot'] for doc in existing_docs_cursor}

# Step 4: Find the difference between the set of all IDs and the set of IDs found in the DB.
missing_ot_ids = set(all_ids) - ids_in_db

# Print the missing IDs
for current_id in missing_ot_ids:
    print(f" [X] The ot with id: {current_id} is not in the Database ")

# Your final result is a list of the missing IDs
missing_ot = list(missing_ot_ids)


 [X] The ot with id: 52335 is not in the Database 


In [31]:
del obj_lists_dask

#### Existen OTs en estado PDF repetidas en las carpetas

Esta es la razon de que no coincidan los numeros

In [17]:
# Buscando en MongoDB con REGEX:

csv = "/home/vlad/Documents/temp_borrar/a-reporete_del_reporte/2024-reportes/ots_query_mongo_2-24/eerssa.ot_v22 OTS del 2024.csv"

#Load file into Pandas
csv_df = pd.read_csv( csv )
csv_df.head()

regex_id = csv_df["id_ot"].tolist()
len(regex_id)

4293

In [24]:
ids_not_pdf = set(all_ids) - set(regex_id)
len(ids_not_pdf)

3

In [28]:
print(f" Total Elementos en all_ids : {len(all_ids)} Elementos unicos : {len(set(all_ids))}")

 Total Elementos en all_ids : 4316 Elementos unicos : 4296


In [30]:
from collections import Counter

pdf_contador  = Counter(all_ids)

repeated_items = {item: count for item, count in pdf_contador.items() if count > 1 }
print("\nRepeated items and their counts:")
pprint(repeated_items)


Repeated items and their counts:
{122020: 3,
 124079: 2,
 125318: 2,
 125661: 2,
 125698: 2,
 125755: 2,
 125796: 2,
 125860: 2,
 125945: 2,
 126011: 2,
 126083: 2,
 129097: 2,
 129152: 2,
 129161: 2,
 129167: 2,
 129927: 2,
 131619: 3,
 134113: 2}


### NUEVO DELTA LAKE Descargar todo el 2025

Para iniciar crearemos un DeltaLake de las OTs del 2025

In [ ]:
import re
# Assuming CurrentCollection is a valid pymongo.collection.Collection object
# and is already connected to your database as in consumer.py.

regex_pattern = re.compile("^2025")
query_filter = {"fecha": regex_pattern}

# Use find() to get a cursor that points to all matching documents
cursor = CurrentCollection.find(query_filter)

obj_list = []
for document in cursor:
  ot = OrdenTrabajo.GestionOt.from_dict( document )
  obj_list.append( Actividades.ConvertirOT_a_ActividadesCSV(ot))
  #print(f"Processing document with id_ot: {document.get('id_ot')}")
df = pd.concat(obj_list, ignore_index=True)
